# Jira Issue CSV Exporter
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Sorting, Strings · **Difficulty/Frequency:** Popular! (10/10)


## Concepts

**What this problem is really testing:**
- CSV escaping rules — a small set of "if this character appears, do this" rules
- Building strings efficiently (not slowly, by accident)
- Sorting by more than one column, each with its own direction
- Generators (`yield`) — producing results one at a time instead of all at once

**Why each one shows up here:**
- The main task is just *safe serialization* — every field has to survive being written into a comma-separated format without breaking it. So this is really a string-handling problem, not a data-structure problem.
- Follow-up 1 (custom sort order) needs a **stable, multi-column sort**.
- Follow-up 2 (streaming) needs a **generator**, so we can hand back rows one at a time instead of building one giant string.

**The one idea to hold onto:** everything routes through a single helper, `escape_field`. That one function decides how to safely write a value. Sorting and streaming don't change that function at all — they just change *the order* and *the pace* at which it gets called.

---

### Quick primers — the building blocks used below

**CSV escaping (a rule, not a data structure).**
CSV only cares about three characters:
- `,` — separates fields
- `"` — used for escaping
- `\n` — separates rows

Rule: if a field contains any of those three characters, do two things, **in this order**:
1. Replace every `"` inside the field with `""`
2. Wrap the whole field in `"..."`

Do it in the other order and you'll accidentally double the wrapping quotes too — order matters here.

**Building strings the fast way (mutable vs. immutable).**
- In Python, strings can't be changed in place. `s = s + more` actually creates a **brand new string** and copies everything into it.
- Doing this over and over in a loop (`s += chunk`, N times) can silently become **O(N²)** — slow, and it gets worse as N grows.
- The safe habit: collect all the pieces in a **list**, then call `"".join(pieces)` **once** at the end. This is **O(total length)** — i.e., linear, and always fast regardless of language or implementation details.

**What is a generator?**
- A generator is a function that uses `yield` instead of (or alongside) `return`.
- Calling it doesn't run the code right away — it hands you back a generator object that runs a little bit at a time, each time you ask for the next value (via `next()` or a `for` loop).
- **Why it matters:** producing one row costs whatever that row costs (here, O(F) for F fields) — but the generator itself only ever holds **O(1)** extra memory, no matter how many rows exist in total. Compare that to building the whole thing up front, which needs **O(N·F)** memory.
- **In Python:** write a function with `yield` inside it, or use a short generator expression like `(x for x in iterable)`.

**Sorting by multiple columns, each with its own direction.**
- Python's `sorted()` uses an algorithm called **Timsort**, which is **stable** — meaning: items that are already "equal" under the sort key keep their original relative order.
- That stability is the trick that lets you sort by several columns with *different* directions (some ascending, some descending), without building one big combined key.
- **How:** sort by the **least important column first**, then the next one, and finish with the **most important column last**. Because each sort is stable, the earlier sorting doesn't get lost — it just becomes the tiebreaker underneath the final sort.
- **Cost:** each `sorted()` call is O(N log N). Doing this once per sort key (K keys total) costs O(K · N log N), and each call also creates a new list of size O(N).


## Problem Statement

Implement a function that takes a list of Jira **issues** (dicts) and a list of **field names** (columns), and produces a valid CSV string.

**Example**

```python
fields = ["id", "summary", "status", "assignee"]

issues = [
    {"id": "PROJ-1", "summary": "Fix login bug",       "status": "Done",        "assignee": "alice"},
    {"id": "PROJ-2", "summary": "Add, export feature",  "status": "In Progress", "assignee": "bob"},
    {"id": "PROJ-3", "summary": 'He said "hello"',      "status": "Todo",        "assignee": None},
]
```

```csv
id,summary,status,assignee
PROJ-1,Fix login bug,Done,alice
PROJ-2,"Add, export feature",In Progress,bob
PROJ-3,"He said ""hello""",Todo,
```

**CSV rules**
- First row is the header, using `fields` in order.
- A field containing a comma, a double quote, or a newline is wrapped in double quotes; internal `"` becomes `""`.
- `None` and missing keys both export as an empty string.
- Rows are joined with `\n`.

**Constraints:** `fields` has at least one entry; `issues` may be empty (return just the header row).


### Approach 1 — Naive (no escaping)

**Idea:** the first instinct — for each issue, look up every field and join with commas; join all rows with `\n`. This is the "does it work on the happy path" version.

**Time complexity:** O(N·F) — F field lookups for each of N issues, same as the correct version below. The naive version is not asymptotically *worse*; it's simply **wrong**.

**Space complexity:** O(N·F) for the output.

Watch what happens on `PROJ-2` (a comma inside `summary`) and `PROJ-3` (a literal `"` inside `summary`, and a `None` assignee) — both corrupt the CSV structure.


In [1]:
from typing import Any, Dict, List


def export_to_csv_naive(fields: List[str], issues: List[Dict[str, Any]]) -> str:
    """First-draft exporter: no quoting, no escaping. Breaks on real data."""
    lines = [",".join(fields)]
    for issue in issues:
        # str(None) would literally print "None" -- and a raw comma/quote
        # inside a value silently corrupts the row structure below.
        row = [str(issue.get(f, "")) for f in fields]
        lines.append(",".join(row))
    return "\n".join(lines)


In [2]:
fields = ["id", "summary", "status", "assignee"]

issues = [
    {"id": "PROJ-1", "summary": "Fix login bug",       "status": "Done",        "assignee": "alice"},
    {"id": "PROJ-2", "summary": "Add, export feature",  "status": "In Progress", "assignee": "bob"},
    {"id": "PROJ-3", "summary": 'He said "hello"',      "status": "Todo",        "assignee": None},
]

res = export_to_csv_naive(fields, issues)
res

'id,summary,status,assignee\nPROJ-1,Fix login bug,Done,alice\nPROJ-2,Add, export feature,In Progress,bob\nPROJ-3,He said "hello",Todo,None'

### Approach 2 — Optimal (escape-safe)

**Idea:** isolate the CSV quoting rule in one helper, `escape_field`, and route the header and every row value through it. A field needs quoting if it contains a comma, a double quote, or a newline; when it does, double every internal `"` **first**, then wrap in `"..."`. `None` and missing keys both collapse to `""` before that check ever runs.

**Time complexity:** O(F + N·F) — the header costs O(F), each of N rows costs O(F) field lookups, and `escape_field` is O(len(value)) per field, which sums to the total output length.

**Space complexity:** O(F + N·F) for the output string, plus O(F) transient working space per row (the list comprehension building one row).


In [3]:
from typing import Any, Dict, List, Optional


def escape_field(value: Any) -> str:
    """Apply the CSV quoting rule to a single value.

    None -> "". Otherwise: quote (and double internal ") iff the value
    contains a comma, a double quote, or a newline.
    """
    if value is None:
        return ""
    s = str(value)
    if any(c in s for c in [",", '"', "\n"]):
        s = s.replace('"', '""')   # double internal quotes BEFORE wrapping,
        return f'"{s}"'            # or you'd corrupt the wrapper quotes too
    return s


def export_to_csv(fields: List[str], issues: List[Dict[str, Any]]) -> str:
    """Correct exporter: every field -- header or row -- goes through escape_field."""
    lines = [",".join(escape_field(f) for f in fields)]     # header
    print("lines ",lines)
    for issue in issues:
        row = [escape_field(issue.get(f, "")) for f in fields]  # missing key -> "" -> escape_field("") -> ""
        print("rows ",row)
        lines.append(",".join(row))
    return "\n".join(lines)   # list + one join at the end: O(total length), not O(N^2)


In [4]:
fields = ["id", "summary", "status", "assignee"]

issues = [
    {"id": "PROJ-1", "summary": "Fix login bug",       "status": "Done",        "assignee": "alice"},
    {"id": "PROJ-2", "summary": "Add, export feature",  "status": "In Progress", "assignee": "bob"},
    {"id": "PROJ-3", "summary": 'He said "hello"',      "status": "Todo",        "assignee": None},
]

res = export_to_csv(fields, issues)
res

lines  ['id,summary,status,assignee']
rows  ['PROJ-1', 'Fix login bug', 'Done', 'alice']
rows  ['PROJ-2', '"Add, export feature"', 'In Progress', 'bob']
rows  ['PROJ-3', '"He said ""hello"""', 'Todo', '']


'id,summary,status,assignee\nPROJ-1,Fix login bug,Done,alice\nPROJ-2,"Add, export feature",In Progress,bob\nPROJ-3,"He said ""hello""",Todo,'

### Follow-up 1 — Field Ordering (custom multi-key sort)

**Idea:** accept a `sort_spec` of `(field_name, direction)` tuples, e.g. `[("status", "ASC"), ("assignee", "DESC")]`, and sort issues by all of them before exporting — `status` is the primary key, `assignee` breaks ties.

Because directions can be **mixed** (some ASC, some DESC) and values may not be negatable (strings), a single composite key with one global `reverse=` flag doesn't work. Instead: exploit that `sorted()` is **stable** and apply it **once per key, from least significant to most significant** — each pass's stability locks in the ordering from every previous (less significant) pass.

**Time complexity:** O(K · N log N) for K sort keys.

**Space complexity:** O(N) new list per `sorted()` call (K of them); the issues themselves are not copied, only referenced.


In [ ]:
from typing import Any, Dict, List, Sequence, Tuple

SortSpec = Sequence[Tuple[str, str]]   # e.g. [("status", "ASC"), ("assignee", "DESC")]


def sort_issues(issues: List[Dict[str, Any]], sort_spec: SortSpec) -> List[Dict[str, Any]]:
    """Multi-key stable sort with an independent ASC/DESC direction per key."""
    result = list(issues)
    # Apply from LAST key to FIRST: the final pass (the primary key) is the
    # one whose order survives, and stability preserves every earlier pass's
    # tie-breaking underneath it.
    for field, direction in reversed(sort_spec):
        reverse = direction.upper() == "DESC"
        result = sorted(result, key=lambda i, f=field: (i.get(f) is None, i.get(f)), reverse=reverse)
        # `i.get(f) is None` sorts all None/missing values to one end
        # regardless of direction, instead of crashing on None vs str compares.
    return result


def export_to_csv_sorted(fields: List[str], issues: List[Dict[str, Any]],
                          sort_spec: SortSpec = ()) -> str:
    """Same exporter as Approach 2, with an optional pre-sort applied first."""
    ordered = sort_issues(issues, sort_spec) if sort_spec else issues
    return export_to_csv(fields, ordered)


### Follow-up 2 — Streaming Export (generator)

**Idea:** for millions of issues, building one giant string is impractical (peak memory = O(N·F)). Restructure the exact same logic as a **generator**: `yield` the header once, then `yield` one escaped row per issue. The caller drives it with a `for` loop and can write straight to a file, HTTP response, or socket — never holding more than one row in memory.

**Time complexity:** O(F) to produce the header, then O(F) **per row on demand**. Total work to fully drain it is still O(N·F) — streaming doesn't change the total work, only *when* and *how much at once* it happens.

**Space complexity:** O(F) — one row's worth of state alive at any moment, independent of N. (Sorting still needs the whole `issues` list to pick a sort order — Follow-up 1 and Follow-up 2 don't compose for free at true "can't fit in memory" scale; that trade-off is worth naming out loud in an interview.)


In [ ]:
from typing import Any, Dict, Iterator, List


def export_to_csv_stream(fields: List[str], issues: List[Dict[str, Any]]) -> Iterator[str]:
    """Generator version of export_to_csv: yields one CSV line at a time, no trailing newline."""
    yield ",".join(escape_field(f) for f in fields)          # header, produced first
    for issue in issues:                                     # each row computed lazily, on demand
        row = [escape_field(issue.get(f, "")) for f in fields]
        yield ",".join(row)


def write_csv_stream(fields: List[str], issues: List[Dict[str, Any]], fh) -> None:
    """Example consumer: write a streamed export to any file-like object, one line at a time."""
    for line in export_to_csv_stream(fields, issues):
        fh.write(line)
        fh.write("\n")


## Verification

Run every approach against the worked example, plus a few edge cases.

In [ ]:
import io

fields = ["id", "summary", "status", "assignee"]
issues = [
    {"id": "PROJ-1", "summary": "Fix login bug",      "status": "Done",        "assignee": "alice"},
    {"id": "PROJ-2", "summary": "Add, export feature", "status": "In Progress", "assignee": "bob"},
    {"id": "PROJ-3", "summary": 'He said "hello"',    "status": "Todo",        "assignee": None},
]

expected = (
    'id,summary,status,assignee\n'
    'PROJ-1,Fix login bug,Done,alice\n'
    'PROJ-2,"Add, export feature",In Progress,bob\n'
    'PROJ-3,"He said ""hello""",Todo,'
)

# --- Approach 1: naive is expected to get this WRONG ---
naive_out = export_to_csv_naive(fields, issues)
print("Naive output:\n" + naive_out + "\n")
assert naive_out != expected, "naive should NOT match -- it has no escaping"
assert "Add, export feature" in naive_out and '"Add, export feature"' not in naive_out

# --- Approach 2: correct exporter must match exactly ---
correct_out = export_to_csv(fields, issues)
print("Correct output:\n" + correct_out)
assert correct_out == expected

# --- Edge cases: empty issues, missing key, embedded newline ---
assert export_to_csv(fields, []) == "id,summary,status,assignee"
assert export_to_csv(["id", "missing_field"], [{"id": "X-1"}]) == "id,missing_field\nX-1,"
assert escape_field("line1\nline2") == '"line1\nline2"'

# --- Follow-up 1: multi-key sort, mixed directions ---
# Two issues share status="Todo" so the assignee tiebreaker is actually exercised.
sort_issues_data = [
    {"id": "A", "status": "Todo", "assignee": "carol"},
    {"id": "B", "status": "Done", "assignee": "alice"},
    {"id": "C", "status": "Todo", "assignee": "alice"},
]
sort_spec = [("status", "ASC"), ("assignee", "DESC")]
ordered = sort_issues(sort_issues_data, sort_spec)
ordered_ids = [i["id"] for i in ordered]
print("Sorted by status ASC, assignee DESC ->", ordered_ids)
# "Done" sorts before "Todo" (ASC); within "Todo", "carol" > "alice" so it comes first (DESC).
assert ordered_ids == ["B", "A", "C"]

sorted_out = export_to_csv_sorted(fields, issues, sort_spec)
print("\nexport_to_csv_sorted on the worked example:\n" + sorted_out)

# --- Follow-up 2: streaming must produce the same rows as the string version ---
streamed_lines = list(export_to_csv_stream(fields, issues))
assert "\n".join(streamed_lines) == expected
buf = io.StringIO()
write_csv_stream(fields, issues, buf)
assert buf.getvalue() == expected + "\n"   # write_csv_stream adds a trailing newline per line

print("\nAll checks passed.")


## Discussion — the two open follow-up questions

**What if a field value isn't a string?** (ints, bools, dates, nested objects) — `str(value)` already handles ints and bools sensibly (`42` -> `"42"`, `True` -> `"True"`), which is fine for most spreadsheets. It breaks down for anything with a domain-specific text form (a `datetime` you want as `"2024-01-15"` not Python's default `repr`, or a nested dict/list that has no sane single-cell rendering). The clean fix: let the caller pass an optional `field_formatters: Dict[str, Callable]` and check it before falling back to `str()` — shown below.

**What about a field name that doesn't exist on *any* issue?** The current code emits an all-empty column for every row — which is arguably correct (the caller explicitly asked for that column) but wastes space at scale and may look like a bug in the export. Whether to keep it, drop it, or warn is a product decision, not a code one — worth surfacing to the interviewer rather than silently picking one.


In [ ]:
from datetime import date
from typing import Callable


def escape_field_with_formatters(value: Any, formatters: Dict[str, Callable] = None,
                                  field: str = None) -> str:
    """escape_field, but a per-field formatter runs before the string conversion."""
    if value is not None and formatters and field in formatters:
        value = formatters[field](value)
    return escape_field(value)


demo_issue = {"id": "PROJ-9", "points": 5, "urgent": True,
              "due": date(2024, 1, 15), "labels": ["bug", "urgent", "prod"]}
demo_fields = ["id", "points", "urgent", "due", "labels"]

print("Default str() formatting:     ", [escape_field(demo_issue[f]) for f in demo_fields])
formatters = {"labels": lambda tags: ";".join(tags)}   # str(list) would print "['bug', 'urgent', 'prod']"
print("With a custom 'labels' formatter:",
      [escape_field_with_formatters(demo_issue[f], formatters, f) for f in demo_fields])


## Empirical complexity check

Big-O can't be read off the code by itself with full confidence -- it's worth *measuring* it. The trick is the **doubling ratio**: run the optimal exporter on inputs of doubling size `n` and watch how the runtime grows.

| Growth when n doubles | Implies |
|---|---|
| ~1x | constant / logarithmic |
| ~2x | linear, or n log n (close to 2x with a small log factor) |
| ~4x | quadratic |
| ~8x | cubic |

`export_to_csv` is O(F + N·F); with the field count `F` fixed, that's linear in `N` -- doubling `N` should roughly double the time.


In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

FIELDS = ["id", "summary", "status", "assignee"]

def make_worst_case(n):
    # Every summary contains a comma, forcing escape_field's quoting branch
    # on every single field -- the true worst case for the escaping cost.
    issues = [
        {"id": f"PROJ-{i}", "summary": f"Task, number {i}", "status": "Todo", "assignee": "alice"}
        for i in range(n)
    ]
    return (FIELDS, issues)

solutions = {"export_to_csv (optimal)": export_to_csv}
sizes = [2000, 4000, 8000, 16000]
benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Isolate escaping/serialization rules in one helper.** `escape_field` is the single source of truth for the CSV quoting rule; the header and every row value funnel through it, so the rule can't be applied inconsistently. Same idea applies to JSON string escaping, URL encoding, SQL parameter binding.
- **Collapse `None` and "missing key" into the same code path.** `issue.get(field, "")` plus a `None` check inside the shared helper means the call site never special-cases either one.
- **State the escaping predicate explicitly, and get the operation order right.** Double internal quotes *before* wrapping — reversing the order corrupts the wrapper quotes. This "characterize the special cases, then apply the transform in the right order" shape recurs in any escaping/encoding problem.
- **Build large strings with `list.append` + one `"".join()`, never repeated `+=` in a loop.** Immutable strings make repeated concatenation risk O(N²); `join` is the always-safe O(total length) habit.
- **Mixed-direction multi-key sort = repeated stable sort, least significant key first.** No need for a composite key or custom comparator when your sort is stable — sort once per key, in reverse priority order, and stability does the rest.
- **"Return it all at once" vs. "yield it lazily" is a peak-memory decision, not a total-work one.** A generator doesn't reduce the O(N·F) total work of an export — it caps memory at O(F) by producing results on demand instead of materializing them all up front. Reach for this whenever "N could be huge" appears in the prompt.
- **When two follow-ups don't compose for free, say so.** Sorting needs to see the whole input; streaming assumes you don't have to. Naming that tension (pre-sorted input, or an external/out-of-core sort) is itself a strong interview signal.
- **Related problems:** any "serialize to a text format" task (JSON, XML/HTML escaping, SQL query building), "sort by multiple criteria" tasks (e.g. sort intervals by start then length), and "process a huge collection without loading it all" tasks (log processing, paginated API export).
- **Common pitfalls:** wrapping before doubling quotes; treating `None` and `""` differently instead of the same; assuming `str.join` semantics apply to a generator without materializing it first; forgetting that streaming and "need the whole list to sort/dedupe" are in tension.
